# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access fields as attributes, not as dict keys or iteration as per instructions
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's explore the different record sets in the dataset via their `@id`. For each record set, we'll show the available fields/columns by their `@id`.

In [ ]:
# List all record sets and show fields with their @id
print("Available record sets in dataset (by @id):")
record_set_ids = []
for rs in md.recordSet:
    print(f"- RecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    # List fields
    if hasattr(rs, 'field') or 'field' in rs:
        fields = rs.field if hasattr(rs,'field') else rs['field']
        print("  Fields/columns in this RecordSet:")
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    * Field @id: {f['@id']}")
            else:
                print(f"    * Field (string or other reference): {f}")
    else:
        print("  (No fields/columns found)")

# Preview the first two records for each RecordSet (using @id as required)
for rs_id in record_set_ids:
    print(f"\nRecordSet preview for @id={rs_id} (first 2 records):")
    records_iter = dataset.records(record_set=rs_id)
    for i, rec in enumerate(records_iter):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In this example, we'll extract data from all available record sets found in the dataset by their `@id`.

In [ ]:
# Extract data from each record set into a pandas DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    print(f"\nColumns in RecordSet @id={record_set_id}:")
    print(df.columns.tolist())
    print(f"Sample records from RecordSet @id={record_set_id}:")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll demonstrate on the largest/highest-population RecordSet (the first in the list). Numeric fields are chosen by examining the DataFrame.

In [ ]:
# Use the first available record set for demonstration
if len(record_set_ids) == 0:
    raise RuntimeError("No record sets found in the metadata.")
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Try to select a numeric field by inferring from DataFrame dtypes
possible_numeric = df.select_dtypes(include=['float', 'int']).columns.tolist()
if len(possible_numeric) == 0:
    # Try any column with 'molecular' or 'weight' or 'count' or 'abundance'
    for col in df.columns:
        lc = col.lower()
        if any(kw in lc for kw in ['weight', 'count', 'abundance', 'coverage', 'number', 'percent', 'value', 'mw', 'protein']) and pd.api.types.is_numeric_dtype(df[col]):
            possible_numeric.append(col)
if len(possible_numeric) == 0:
    raise RuntimeError('No numeric field found for EDA.')
# Pick the first numeric field found
numeric_field = possible_numeric[0]
print(f"Using numeric field for EDA: {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a categorical or grouping field
non_numeric = [c for c in df.columns if df[c].dtype == object or pd.api.types.is_categorical_dtype(df[c])]
group_field = non_numeric[0] if len(non_numeric) > 0 else None
if group_field:
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data (mean of {numeric_field} by {group_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of the selected numeric field and show a boxplot by the grouping (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field is not None and group_field in df.columns:
    plt.figure(figsize=(12, 5))
    # Pick top 10 frequent categories
    top_groups = df[group_field].value_counts().index[:10]
    filtered = df[df[group_field].isin(top_groups)]
    sns.boxplot(x=group_field, y=numeric_field, data=filtered)
    plt.title(f"{numeric_field} distribution by {group_field} (top 10)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset, sourced from mass spectrometry of extracellular vesicles, contains well-structured record sets identified by Croissant `@id`s.
- Using `mlcroissant`, we programmatically loaded and inspected both metadata and data records by their unique Croissant identifiers.
- The example EDA steps showcased filtering, normalization, grouping, and plotting of numeric fields, all referenced through explicit Croissant `@id` links.
- This workflow provides a reproducible and schema-consistent pathway for further machine learning or statistical analysis of Croissant-packaged datasets.